# The Four Steps of a Hypothesis Test — and How Not to Read One

Everything from the last six notebooks compresses into a four-step checklist: **state** the hypotheses ($H_0$: baseline, $H_1$: what you want to prove) → **design** the test (pick the statistic, pick $\alpha$) → **compute** the observed statistic → **decide** (typically: $p < \alpha$?). Then the part where people actually go wrong — interpretation. Two misconceptions get formally retired: the p-value is **not** the probability that $H_0$ is true (it's the probability of seeing data like yours by chance *if* $H_0$ holds — small p means $H_0$ is a bad model for your data), and failing to reject does **not** mean $H_0$ is true (the spam filter never certifies ham — it only ever lacks evidence of spam). It is based on the seventh video of Lesson 8 (Hypothesis Testing).

## 1. Introduction

A consolidation notebook: the full pipeline as a reusable recipe, the error facts worth keeping fresh, and the interpretation rules that survive contact with real conclusions.

After completing this notebook you will be able to:

- Run the four steps — state, design, compute, decide — on any mean-testing problem.
- Restate the two error types and the entanglement warning: at fixed $n$, a stingy $\alpha$ silently buys a bloated $\beta$.
- Say precisely what a p-value is and is not, and why "probability $H_0$ is true" is not on the menu.
- Translate "fail to reject" into its only honest reading: not enough evidence — for spam, for heights, for every test.
- Demonstrate both misconceptions numerically, not just verbally.

## 2. Theory

### The four steps

1. **State your hypotheses.** The null $H_0$ is the baseline — for the running example, $\mu = 66.7$ inches. The alternative $H_1$ is *usually the statement you wish to prove* — here $\mu > 66.7$.
2. **Design the test.** Decide the statistic you'll work with (the sample mean) and the significance level — most commonly $\alpha = 0.05$. Remember what $\alpha$ is: the **maximum probability of a Type I error**, and it should always be small.
3. **Compute the observed statistic.** For all the previous examples: $\bar{x} = 68.442$.
4. **Make the decision.** The common route: compute the p-value; if $p < \alpha$, reject $H_0$ and accept $H_1$.

### The error facts, kept fresh

| | |
|---|---|
| Type I | rejecting $H_0$ when it was true |
| Type II | failing to reject $H_0$ when $H_1$ was true |
| $\alpha$ | your design parameter = max Type I probability |

**The entanglement warning:** for a fixed sample size, the two error probabilities are entangled. Be mindful when choosing $\alpha$ — making it very small may be quietly forcing a Type II probability that is far too high (notebook 06's see-saw, now as a design caution).

### Misconception 1: what the p-value is not

*"The p-value is the probability that $H_0$ is true."* — **False.** A small p-value does justify rejecting the null, but it is not the null's probability. The p-value is **the probability of seeing data as extreme as yours by chance, assuming $H_0$**. A small p-value says: *the null hypothesis is a bad model for your data* — what you saw would be a rare accident under it. The conditioning runs from hypothesis to data, never the reverse (that direction belongs to Bayesians with priors — Lesson 6).

### Misconception 2: what "fail to reject" is not

*"Not rejecting $H_0$ means $H_0$ is true."* — **Also false.** Recall the spam filter: an email without enough spam evidence is never declared *ham* — the most you can guarantee is that **there was not enough evidence to show it was spam**. The same principle applies to every hypothesis test: non-rejection is an absence of evidence, not evidence of absence. (Notebook 06 quantified why: against nearby alternatives, the test's power can be feeble — failing to reject is then the *expected* outcome even when $H_1$ is true.)

## 3. Worked Examples

### Example 1: the heights test, four steps in four lines

1. $H_0: \mu = 66.7$; $H_1: \mu > 66.7$ (right-tailed — the claim is *increase*).
2. Statistic: $\bar{X}$; level: $\alpha = 0.05$ ($\sigma = 3$ known, $n = 10$).
3. Observed: $\bar{x} = 68.442$.
4. $p = 0.0332 < 0.05$ → reject $H_0$: the data support an increase.

### Example 2: a fresh run — the coffee machine

A machine should dispense 250 ml; is it still calibrated? ($\sigma = 4$ ml known, $n = 16$ cups measured.)

1. $H_0: \mu = 250$; $H_1: \mu \ne 250$ (two-tailed — miscalibration in either direction matters).
2. Statistic $\bar{X}$, $\alpha = 0.05$.
3. Observed: $\bar{x} = 247.8$.
4. $z = (247.8 - 250)/(4/\sqrt{16}) = -2.2$, $p = 2\,\Phi(-2.2) = 0.028 < 0.05$ → reject: the machine drifted.

### Example 3: sentence repair

| broken conclusion | honest version |
|---|---|
| "p = 0.03, so there's a 3% chance heights didn't increase." | "If heights hadn't increased, data this extreme would occur only 3% of the time." |
| "p = 0.4, so the machine is fine." | "The evidence is insufficient to conclude the machine drifted." |
| "We accept $H_0$." | "We fail to reject $H_0$." |
| "The test proves $\mu > 66.7$." | "At $\alpha = 0.05$, the data support rejecting $\mu = 66.7$ in favor of $\mu > 66.7$." |

## 4. Python Examples

First the imports (with the numpy 2.x compatibility shim used throughout this project), then the four steps as one reusable function.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp

# numpy < 2 fallback: np.trapezoid was called np.trapz
if not hasattr(np, "trapezoid"):
    np.trapezoid = np.trapz

rng = np.random.default_rng(77)

Phi = lambda z: 0.5 * (1 + float(sp.erf(sp.Float(z) / sp.sqrt(2))))

def hypothesis_test(data, mu0, sigma, tail, alpha=0.05, label=""):
    """The four steps, narrated: state, design, compute, decide."""
    n = len(data)
    x_bar = np.mean(data)
    z_obs = (x_bar - mu0) / (sigma / np.sqrt(n))
    p = {"right": 1 - Phi(z_obs),
         "left": Phi(z_obs),
         "two": 2 * (1 - Phi(abs(z_obs)))}[tail]
    reject = p < alpha
    sign = {"right": ">", "left": "<", "two": "!="}[tail]
    print(f"--- {label} ---")
    print(f"1. state:   H0: mu = {mu0}   H1: mu {sign} {mu0}")
    print(f"2. design:  statistic X-bar, alpha = {alpha}, sigma = {sigma} known, n = {n}")
    print(f"3. compute: observed x-bar = {x_bar:.3f} (z = {z_obs:.3f})")
    print(f"4. decide:  p = {p:.4f} {'<' if reject else '>='} {alpha} -> "
          f"{'REJECT H0, accept H1' if reject else 'fail to reject H0 (not enough evidence)'}")
    return reject

### The two examples through the pipeline

In [ ]:
heights = np.array([64.12, 71.15, 66.41, 70.17, 73.13,
                    65.10, 69.16, 67.39, 71.97, 65.82])
hypothesis_test(heights, mu0=66.7, sigma=3.0, tail="right",
                label="heights of 18-year-olds")

print()
coffee = np.array([249.1, 246.0, 250.3, 245.7, 248.9, 247.2, 251.0, 244.8,
                   249.6, 246.5, 248.0, 247.3, 250.1, 245.9, 246.8, 247.6])
hypothesis_test(coffee, mu0=250.0, sigma=4.0, tail="two",
                label="coffee machine calibration")

### Misconception 2, demonstrated: a large p proves nothing

Simulate two worlds — one where $H_0$ is true ($\mu = 66.7$) and one where it's false ($\mu = 68$) — and look only at the tests that *failed to reject*. Plenty of false-null worlds hide in that pile: non-rejection cannot certify the null.

In [ ]:
n_sims = 100_000
se = 3.0 / np.sqrt(10)

p_h0 = np.array([1 - Phi(z) for z in (rng.normal(66.7, se, n_sims) - 66.7) / se])
p_h1 = np.array([1 - Phi(z) for z in (rng.normal(68.0, se, n_sims) - 66.7) / se])

not_rejected_h0 = (p_h0 >= 0.05).mean()
not_rejected_h1 = (p_h1 >= 0.05).mean()
print(f"P(fail to reject | H0 true, mu = 66.7)  = {not_rejected_h0:.1%}")
print(f"P(fail to reject | H0 FALSE, mu = 68.0) = {not_rejected_h1:.1%}")
print("\nFailing to reject is the routine outcome in BOTH worlds -- observing it")
print("tells you almost nothing about which world you are in. 'Not enough")
print("evidence' is the only conclusion the outcome supports.")

## 5. Visualizations

### The pipeline at a glance

In [ ]:
fig, ax = plt.subplots(figsize=(11, 3.6))
steps = [
    ("1. STATE", "$H_0$: baseline\n$H_1$: the claim\nto prove"),
    ("2. DESIGN", "choose statistic\nchoose $\\alpha$\n(before data!)"),
    ("3. COMPUTE", "observed statistic\nfrom the sample\n($\\bar{x}$ = 68.442)"),
    ("4. DECIDE", "p < $\\alpha$: reject\notherwise: not\nenough evidence"),
]
for i, (title, body) in enumerate(steps):
    x = i * 2.7
    box = plt.Rectangle((x, 0), 2.2, 1.9, facecolor="tab:blue", alpha=0.12,
                        edgecolor="tab:blue", linewidth=1.5)
    ax.add_patch(box)
    ax.annotate(title, xy=(x + 1.1, 1.55), ha="center", fontsize=11,
                fontweight="bold", color="tab:blue")
    ax.annotate(body, xy=(x + 1.1, 0.75), ha="center", va="center", fontsize=9)
    if i < 3:
        ax.annotate("", xy=(x + 2.65, 0.95), xytext=(x + 2.25, 0.95),
                    arrowprops=dict(arrowstyle="->", color="black", lw=1.5))
ax.set_xlim(-0.3, 11.2)
ax.set_ylim(-0.25, 2.3)
ax.axis("off")
ax.set_title("Hypothesis testing in four steps -- steps 1-2 happen before any data",
             fontsize=11)
plt.tight_layout()
plt.show()

### Why a large p can't confirm $H_0$

The p-value distributions of the two simulated worlds, with the not-rejected region shaded: both worlds pour probability into it.

In [ ]:
plt.figure(figsize=(10, 4.6))
bins = np.linspace(0, 1, 41)
plt.hist(p_h0, bins=bins, density=True, color="tab:green", alpha=0.55,
         label=r"world A: $H_0$ true ($\mu$ = 66.7)")
plt.hist(p_h1, bins=bins, density=True, color="tab:red", alpha=0.55,
         label=r"world B: $H_0$ false ($\mu$ = 68)")
plt.axvline(0.05, color="black", linestyle="--", linewidth=1.4)
plt.annotate("reject\n(p < 0.05)", xy=(0.02, 5.5), ha="center", fontsize=9)
plt.annotate("fail to reject -- both worlds live here comfortably",
             xy=(0.5, 5.5), ha="center", fontsize=10)
plt.xlabel("p-value")
plt.ylabel("density")
plt.title("A large p-value is compatible with a true AND a false null")
plt.grid(True, alpha=0.3)
plt.legend(fontsize=9)
plt.show()

## 6. Exercises

### Basic

**Exercise 1.** Sort these activities into the four steps: computing $\bar{x}$; choosing $\alpha = 0.01$; writing $H_1: \mu < 3$; comparing $p$ to $\alpha$; deciding to use the sample mean; declaring "not enough evidence."

**Exercise 2.** Verdict each conclusion (valid / misconception 1 / misconception 2): (a) "p = 0.02: there's a 2% chance the null is true"; (b) "p = 0.02: data this extreme arises 2% of the time under the null"; (c) "p = 0.31, so the mean is still 66.7"; (d) "p = 0.31: the evidence doesn't establish an increase."

### Intermediate

**Exercise 3.** Run the four-step pipeline on this delivery-time sample (target: has the mean dropped below 3.0 days? $\sigma = 0.8$ known): `[2.4, 3.1, 2.2, 2.9, 2.5, 2.8, 2.1, 2.7, 3.0, 2.3, 2.6, 2.2]`. Write the honest one-sentence conclusion.

**Exercise 4.** *The entanglement warning, quantified.* For the heights design ($n = 10$), compute $\beta(68)$ at $\alpha = 0.05$ and at $\alpha = 0.001$. Then find how large $n$ must be so that even the strict $\alpha = 0.001$ keeps $\beta(68) \le 0.10$.

### Advanced

**Exercise 5.** *p-value vs P($H_0$ | data).* Give the two worlds prior probability 50/50 ($\mu = 66.7$ vs $\mu = 68$, $\sigma = 3$, $n = 10$) and compute the genuine posterior $P(H_0 \mid \bar{x} = 68.442)$ via Bayes. Compare it to the p-value 0.0332 and explain why the two numbers answer different questions (and would diverge further with a different prior).

**Exercise 6.** *When does "fail to reject" carry weight?* The demonstration above showed non-rejection is uninformative at $n = 10$. Repeat it at $n = 100$ and $n = 1000$: how does $P(\text{fail to reject} \mid \mu = 68)$ change, and what (power-based) condition makes a non-rejection actually persuasive evidence that any change is small?

## 7. Solutions

In [ ]:
# Exercise 1: sorting into steps
print("step 1 (state):   writing H1: mu < 3")
print("step 2 (design):  choosing alpha = 0.01; deciding to use the sample mean")
print("step 3 (compute): computing x-bar")
print("step 4 (decide):  comparing p to alpha; declaring 'not enough evidence'")

In [ ]:
# Exercise 2: verdicts
print("(a) MISCONCEPTION 1 -- the p-value is not P(H0 true).")
print("(b) VALID -- the definition itself.")
print("(c) MISCONCEPTION 2 -- non-rejection never establishes the null.")
print("(d) VALID -- the honest reading of a large p.")

In [ ]:
# Exercise 3: delivery times through the pipeline
deliveries = np.array([2.4, 3.1, 2.2, 2.9, 2.5, 2.8, 2.1, 2.7, 3.0, 2.3, 2.6, 2.2])
hypothesis_test(deliveries, mu0=3.0, sigma=0.8, tail="left",
                label="delivery times")
print("\nConclusion sentence: 'At the 5% significance level, the data support the")
print("claim that mean delivery time has dropped below 3.0 days.'")

In [ ]:
# Exercise 4: the entanglement, quantified
def z_quantile(p):
    return float(sp.sqrt(2) * sp.erfinv(2 * sp.Float(p) - 1))

def beta68(alpha, n):
    se_ = 3.0 / np.sqrt(n)
    k = 66.7 + z_quantile(1 - alpha) * se_
    return Phi((k - 68.0) / se_)

for a in (0.05, 0.001):
    print(f"alpha = {a:>6}: beta(68) at n = 10 is {beta68(a, 10):.3f}")
n = 10
while beta68(0.001, n) > 0.10:
    n += 1
print(f"\nstrict alpha = 0.001 with beta(68) <= 0.10 requires n = {n}")
print(f"The warning in numbers: at n = 10, demanding alpha = 0.001 buys a "
      f"{beta68(0.001, 10):.0%}")
print("chance of MISSING a real 1.3-inch increase. Tiny alphas need big samples.")

In [ ]:
# Exercise 5: posterior vs p-value
se = 3.0 / np.sqrt(10)
x_obs = 68.442
lik_h0 = np.exp(-0.5 * ((x_obs - 66.7) / se) ** 2)
lik_h1 = np.exp(-0.5 * ((x_obs - 68.0) / se) ** 2)
posterior_h0 = lik_h0 * 0.5 / (lik_h0 * 0.5 + lik_h1 * 0.5)
print(f"P(H0 | x-bar = 68.442) with a 50/50 prior = {posterior_h0:.3f}")
print(f"the p-value                                = 0.0332")
print("\nDifferent numbers because different questions: the posterior asks 'how")
print("probable is H0 given the data (and my prior)?'; the p-value asks 'how")
print("surprising is the data given H0?'. Only the Bayesian version may speak of")
print("H0's probability -- and its answer moves with the prior, which the")
print("p-value never uses.")

In [ ]:
# Exercise 6: when non-rejection means something
for n in (10, 100, 1000):
    se_n = 3.0 / np.sqrt(n)
    k_n = 66.7 + z_quantile(0.95) * se_n
    beta_n = Phi((k_n - 68.0) / se_n)
    print(f"n = {n:>5d}: P(fail to reject | mu = 68) = {beta_n:.4f}")
print("\nAt n = 100 a real one-inch increase is almost never missed, and at")
print("n = 1000 essentially never. Rule: a non-rejection is persuasive only when")
print("the test had HIGH POWER against every alternative you care about -- then")
print("'we failed to reject' really does imply 'any change is likely small'.")

### The whole story on one page

Everything above in a single sheet: the four steps run on the heights and the coffee machine, the error facts with the α–β entanglement, both misconceptions in numbers (p-value histograms in a true-null and a false-null world, p-value vs P(H₀ | data)), and the sentence repairs.

<img src="img/07_steps_and_interpretation.png" alt="Concept sheet for this notebook" width="900">

## 8. Conclusion

- The pipeline: **state** ($H_0$ baseline, $H_1$ the claim to prove) → **design** (statistic + $\alpha$, chosen before data; $\alpha$ = max Type I probability, kept small) → **compute** (the observed statistic) → **decide** ($p < \alpha$ → reject).
- Keep the error facts fresh — and respect the entanglement: at fixed $n$, a stingy $\alpha$ can force a Type II probability that's far too high (at $n = 10$, $\alpha = 0.001$ means missing a real 1.3-inch increase 96% of the time).
- **Misconception 1 retired:** the p-value is the probability of data like yours *assuming* $H_0$ — a small one says the null is a bad model for the data; it is never the probability that $H_0$ is true (that question needs a prior).
- **Misconception 2 retired:** failing to reject guarantees only "not enough evidence" — the spam filter never certifies ham, and simulation shows a large p is comfortable in both true-null and false-null worlds. A non-rejection persuades only when the test had high power.

Next: the tools for the realistic case — testing means when $\sigma$ is unknown, with the t-distribution back on stage.